# SujetoLugar · Del ejemplo de entrenamiento a la orden de Unreal

Abre este cuadernillo con el kernel **.venv-decision** y ejecuta las celdas en orden. Entrenarás en CPU los dos clasificadores de la variante, comprobarás qué errores cometen y podrás escribir órdenes para un soldado o para su pelotón. Los pesos de práctica se guardan en una carpeta temporal. Puedes experimentar con los ejemplos sin alterar la partida.

In [1]:
from pathlib import Path
from collections import Counter
from tempfile import TemporaryDirectory
import json
import sys
import joblib
import numpy as np
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

raiz = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "AI NLP/yell").is_dir())
variante = raiz / "Variantes/SujetoLugar"
sys.path[:0] = [str(variante), str(raiz / "AI NLP")]
from slots import SlotClassifier, MOVEMENT, STATIC
from yell.decision_data import build, read_data
from yell.decision import features, wire
from yell.contract import UNITS

## 1. Cómo se construye el dataset

El generador `decision_data.build` reúne plantillas de órdenes de la primera fase, expresiones coloquiales como «{s}, cuidame {z}» y ejemplos de conversación, negación y órdenes incompletas. **{s}** representa al destinatario, **{z}** al lugar y **{t}** al paciente en ejemplos médicos. Cada plantilla literal es una familia. Por etiqueta, el generador mezcla las familias con semilla 421 y las reparte aproximadamente en 70 % para entrenamiento, 15 % para validación y 15 % para prueba.

Después rellena cada plantilla hasta diez veces, alternando nombres, pelotones y cinco zonas. Algunas filas usan «acá» y guardan su referencia en `here`. Elimina repeticiones de texto dentro de la misma plantilla. Así, las variaciones de una plantilla quedan juntas. La división por familias puede conservar expresiones parecidas entre particiones.

La siguiente celda reconstruye el corpus en memoria y lo compara con las filas JSONL que usa el entrenamiento. Observa cuánto crece el número de frases al rellenar las plantillas.

In [2]:
datos = read_data()
reconstruido = build()
for parte, filas in datos.items():
    familias = {f["family"] for f in filas}
    assert filas == reconstruido[parte], f"El archivo de {parte} difiere del generador"
    print(f"{parte:5}  {len(filas):4} filas  {len(familias):3} familias")
    print(dict(sorted(Counter(f["label"] for f in filas).items())))

train  1318 filas  224 familias
{'atacar': 130, 'avanzar': 140, 'cancelar_orden': 10, 'charla': 248, 'curar': 90, 'defender': 130, 'explorar': 130, 'mantener_posicion': 10, 'patrullar': 130, 'reagruparse': 130, 'reanimar': 70, 'retirarse': 100}
val     298 filas   51 familias
{'atacar': 30, 'avanzar': 30, 'cancelar_orden': 10, 'charla': 38, 'curar': 20, 'defender': 30, 'explorar': 30, 'mantener_posicion': 10, 'patrullar': 30, 'reagruparse': 30, 'reanimar': 20, 'retirarse': 20}
test    314 filas   51 familias
{'atacar': 30, 'avanzar': 30, 'cancelar_orden': 10, 'charla': 54, 'curar': 20, 'defender': 30, 'explorar': 30, 'mantener_posicion': 10, 'patrullar': 30, 'reagruparse': 30, 'reanimar': 10, 'retirarse': 30}


## 2. De una frase concreta a una entrada reutilizable

Una fila conserva el texto original, la plantilla, la etiqueta de acción y la respuesta esperada. También incluye el catálogo que estaba disponible al decir la frase. `roster` relaciona cada ID de soldado con su nombre y `aliases` permite reconocer varias maneras de llamar a una unidad. El corpus histórico incluye nombres distintos de los que aparecen en la partida actual.

La función compartida `features` encuentra nombres y lugares en ese catálogo. Para el clasificador reemplaza sus menciones por **unidad** y **lugar**. Por separado devuelve los IDs reconocidos. De este modo se puede aprender la expresión «cuidame» y reutilizarla cuando el jugador nombra a otro soldado.

In [3]:
fila = next(f for f in datos["train"] if f["label"] in MOVEMENT and f["here"] and f["subject"] in f["roster"])
print(json.dumps(fila, indent=2, ensure_ascii=False))
frase_codificada, sujetos, lugares = features(fila["text"], fila["aliases"], fila["here"])
print("Entrada del clasificador:", frase_codificada)
print("Entidades para Unreal:", sujetos, lugares)

{
  "text": "necesito a Alex en acá",
  "family": "necesito a {s} en {z}",
  "label": "avanzar",
  "encoded": "necesito a unidad en lugar",
  "subject": "Alfa-1",
  "location": "base_enemiga",
  "target": null,
  "aliases": {
    "Alfa-1": [
      "Alex"
    ],
    "Alfa-2": [
      "Juan"
    ],
    "Alfa-3": [
      "Ana"
    ],
    "Alfa-4": [
      "Rosa"
    ],
    "Bravo": [
      "bravo"
    ],
    "Médico": [
      "médico"
    ]
  },
  "roster": {
    "Alfa-1": {
      "nombre": "Alex"
    },
    "Alfa-2": {
      "nombre": "Juan"
    },
    "Alfa-3": {
      "nombre": "Ana"
    },
    "Alfa-4": {
      "nombre": "Rosa"
    }
  },
  "here": "base_enemiga"
}
Entrada del clasificador: necesito a unidad en lugar
Entidades para Unreal: ['Alfa-1'] ['base_enemiga']


## 3. Qué aprende cada clasificador

El primer clasificador responde «¿esta frase pide un movimiento?». Ocho acciones del dataset cuentan como positivas. Charla, curación, reanimación y cancelación cuentan como negativas para esa pregunta. Por ejemplo, enviar médicos a curar tiene su propia mecánica en el perfil completo y aquí sería un error convertirlo en desplazamiento.

El segundo clasificador se entrena solo con peticiones de movimiento. Aprende la postura **mantener** para defender o mantener posición, y **avanzar** para las demás. La partida usa esta predicción cuando se activa la opción de postura.

Seguimos la receta de `slots.fit`: deduplicamos pares de frase codificada y etiqueta, añadimos los refuerzos de `frases.json` y excluimos refuerzos cuya cadena coincide literalmente con una entrada reservada. Si amplías los datos, añade paráfrasis positivas y frases parecidas con intención distinta. Por ejemplo, compara una petición de ir al centro con una pregunta sobre quién está en el centro.

In [4]:
entrenamiento = {(f["encoded"], f["label"]) for f in datos["train"]}
inicial = len(entrenamiento)
reservadas = {f["encoded"] for parte in ("val", "test") for f in datos[parte]}
extras = json.loads((variante / "frases.json").read_text(encoding="utf8"))
entrenamiento.update((p, etiqueta) for etiqueta, frases in extras.items() for p in frases if p not in reservadas)
movimientos = [(p, e) for p, e in entrenamiento if e in MOVEMENT]
print("Pares originales / finales:", inicial, len(entrenamiento))
print("Movimiento / resto:", len(movimientos), len(entrenamiento) - len(movimientos))
print("Mantener / avanzar:", sum(e in STATIC for _, e in movimientos), sum(e not in STATIC for _, e in movimientos))

Pares originales / finales: 354 395
Movimiento / resto: 219 176
Mantener / avanzar: 36 183


## 4. Entrenar en CPU

Cada modelo combina dos representaciones TF-IDF. Una cuenta palabras y pares de palabras, de modo que «hacete cargo» puede ser un rasgo distinto de «cargo». La otra cuenta fragmentos de dos a cinco caracteres dentro de cada palabra. Esto permite compartir evidencia entre algunas variantes ortográficas. TF-IDF pondera cada rasgo según su frecuencia en la frase y en el corpus de ajuste.

La regresión logística aprende un peso por rasgo y los combina para estimar la probabilidad de la clase positiva. `C=8` controla la regularización inversa y `class_weight="balanced"` da más peso a la clase menos frecuente. Cada llamada a `fit` aprende su propio vocabulario, sus ponderaciones TF-IDF y sus coeficientes.

In [5]:
def entrenar_binario(filas, positivas):
    textos, objetivos = zip(*sorted((texto, int(etiqueta in positivas)) for texto, etiqueta in filas))
    rasgos = FeatureUnion([
        ("palabras", TfidfVectorizer(ngram_range=(1, 2))),
        ("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))),
    ])
    modelo = make_pipeline(rasgos, LogisticRegression(C=8, class_weight="balanced", random_state=421))
    return modelo.fit(textos, objetivos)

intencion = entrenar_binario(entrenamiento, MOVEMENT)
postura = entrenar_binario(movimientos, STATIC)
for nombre, modelo in [("Movimiento", intencion), ("Mantener", postura)]:
    pesos = modelo[-1].coef_[0]
    nombres = modelo[0].get_feature_names_out()
    print(nombre, "· rasgos con mayor peso positivo:", nombres[np.argsort(pesos)[-6:][::-1]].tolist())

Movimiento · rasgos con mayor peso positivo: ['palabras__lugar', 'palabras__unidad anda', 'palabras__unidad', 'caracteres__r ', 'caracteres__ar ', 'caracteres__ar']
Mantener · rasgos con mayor peso positivo: ['palabras__unidad quedense', 'palabras__quedense', 'palabras__cubran lugar', 'palabras__sostengan la', 'palabras__sostengan', 'palabras__unidad sostengan']


## 5. Elegir el umbral con validación

La probabilidad de movimiento necesita un punto de corte. Un umbral bajo recupera más órdenes y también puede mover una unidad durante una charla. La variante asigna +1 a cada acierto, −3 al movimiento indebido y 0 a la orden omitida. Calculamos esa utilidad en validación para siete umbrales. En caso de empate se elige el primero. Esta calibración evalúa la parte estadística antes de aplicar las reglas del juego.

In [6]:
validacion = datos["val"]
verdad = np.array([f["label"] in MOVEMENT for f in validacion])
prob = intencion.predict_proba([f["encoded"] for f in validacion])[:, 1]
utilidades = {}
print("umbral   aciertos   movimientos indebidos   órdenes omitidas   utilidad")
for corte in (.50, .55, .60, .65, .70, .75, .80):
    tn, fp, fn, tp = confusion_matrix(verdad, prob >= corte, labels=[False, True]).ravel()
    utilidades[corte] = int(tn + tp - 3 * fp)
    print(f"{corte:6.2f} {tn + tp:10} {fp:21} {fn:17} {utilidades[corte]:10}")
umbral = max(utilidades, key=utilidades.get)
print("Umbral elegido:", umbral)

umbral   aciertos   movimientos indebidos   órdenes omitidas   utilidad
  0.50        275                    23                 0        206
  0.55        275                    23                 0        206
  0.60        264                    22                12        198
  0.65        258                    22                18        192
  0.70        244                    22                32        178
  0.75        240                    22                36        174
  0.80        240                    22                36        174
Umbral elegido: 0.5


## 6. Guardar, recargar y conectar con la lógica de la partida

El archivo Joblib contiene, en orden, el clasificador de movimiento, el de postura y el umbral. Guardaremos esa misma estructura en una carpeta temporal y la recargaremos. La clase de práctica hereda el método `classify` que usa Unreal a través del servicio Python. Su constructor permite probar los pesos que acabas de entrenar.

En ejecución, las reglas comprueban sujeto, destino y restricciones como preguntas o negaciones. Una orden explícita reconocida por la gramática puede resolverse directamente. En las demás frases admisibles decide el modelo de movimiento. En el perfil simple el adaptador envía «avanzar» para un destino ordinario y «atacar» para la base rival, porque Unreal requiere esa acción para dañar su núcleo. Una puntuación de 1 también puede venir de una regla.

In [7]:
carpeta_practica = TemporaryDirectory(prefix="sujeto-lugar-")
archivo = Path(carpeta_practica.name) / "intencion.joblib"
joblib.dump((intencion, postura, umbral), archivo, compress=3)

class ClasificadorDePractica(SlotClassifier):
    def __init__(self, archivo, usar_postura=False):
        self.gate, self.stance, self.threshold = joblib.load(archivo)
        self.posture = usar_postura

simple = ClasificadorDePractica(archivo)
con_postura = ClasificadorDePractica(archivo, usar_postura=True)
print("Pesos recargados:", archivo.name, "· umbral:", simple.threshold)

Pesos recargados: intencion.joblib · umbral: 0.5


## 7. Evaluar la decisión completa sobre prueba

Una orden cuenta como correcta cuando coinciden el ID del destinatario y el destino. Para un ejemplo fuera de movimiento, el resultado esperado es charla. Evaluamos también la postura y mostramos errores reales para distinguir una orden omitida de un movimiento indebido. Estos resultados corresponden a **texto sintético**; la transcripción del micrófono introduce otra fuente de error.

In [8]:
def evaluar(modelo):
    resumen, errores = Counter(), []
    for f in datos["test"]:
        salida, _, _ = modelo.classify(None, f["text"], f["aliases"], f["roster"], f["here"], False)
        esperado = (f["subject"], f["location"]) if f["label"] in MOVEMENT else None
        obtenido = (salida["sujeto"]["id"], salida["lugar"]) if salida["tipo"] == "accion" else None
        estado = ("acierto" if esperado == obtenido else "movimiento indebido" if esperado is None
                  else "orden omitida" if obtenido is None else "sujeto o lugar incorrecto")
        resumen[estado] += 1
        if estado != "acierto":
            errores.append((estado, f["text"], esperado, obtenido))
        if modelo.posture:
            deseada = ("mantener" if f["label"] in STATIC else "avanzar") if esperado else None
            resumen["aciertos incluida postura"] += int(esperado == obtenido and salida.get("postura") == deseada)
    return resumen, errores

for nombre, modelo in [("Simple", simple), ("Con postura", con_postura)]:
    resumen, errores = evaluar(modelo)
    print(nombre, "sobre", len(datos["test"]), "frases:", dict(resumen))
    for categoria in dict.fromkeys(e[0] for e in errores):
        print(next(e for e in errores if e[0] == categoria))

Simple sobre 314 frases: {'acierto': 300, 'orden omitida': 10, 'movimiento indebido': 4}
('orden omitida', 'Lucía, los quiero juntos en un mismo sitio en el flanco izquierdo', ('Alfa-1', 'flanco_izquierdo'), None)
('movimiento indebido', 'Lucía, hacen falta curaciones en el flanco izquierdo', None, ('Alfa-1', 'flanco_izquierdo'))


Con postura sobre 314 frases: {'acierto': 300, 'aciertos incluida postura': 269, 'orden omitida': 10, 'movimiento indebido': 4}
('orden omitida', 'Lucía, los quiero juntos en un mismo sitio en el flanco izquierdo', ('Alfa-1', 'flanco_izquierdo'), None)
('movimiento indebido', 'Lucía, hacen falta curaciones en el flanco izquierdo', None, ('Alfa-1', 'flanco_izquierdo'))


## 8. Qué ocurre al hablar

| Componente | Trabajo durante la partida | Origen de su comportamiento |
|---|---|---|
| Whisper Base y Small | Base transcribe el WAV. El servicio puede pedir una segunda lectura a Small si hay incertidumbre. | Reconocedores preentrenados. La variante puede usar CUDA. |
| Sujeto y lugar | Resuelve nombres, alias, zonas y «acá» usando la plantilla enviada por Unreal. | Catálogo y reglas de extracción. |
| Movimiento y postura | Clasifican la frase codificada cuando corresponde. | Los dos modelos supervisados entrenados en este cuadernillo. |
| Gemma 2 | Redacta la respuesta de charla con el contexto del soldado. | Modelo preentrenado de instrucciones con un mensaje de sistema. |
| Kokoro | Convierte esa respuesta en audio con la voz asignada al soldado. | Sintetizador preentrenado. |

Puedes seguir una transmisión real en `WarCommanderDemo/Saved/YellOrders.jsonl`. Compara `stt_primary_ms`, `stt_retry_ms` y `nlp_ms` para distinguir la espera al transcribir de la espera al decidir.

## 9. Prueba tu propia frase

Cambia **frase**, **aqui** o **usar_postura** y vuelve a ejecutar esta celda. Prueba «Juan, avancen al centro», «Alfa, avancen al centro», «Bravo, defiendan acá» y «Juan, ¿cómo estás?». Fíjate en cuándo cambia el ID del destinatario, cuándo interviene la postura y qué JSON recibe el juego.

In [9]:
frase = "Juan, avancen al centro"
aqui = "flanco_derecho"
usar_postura = False

roster = {"Alfa-1": {"nombre": "Alex"}, "Alfa-2": {"nombre": "Juan"}}
aliases = {k: UNITS[k] for k in ("Alfa", "Bravo", "Médico")}
aliases.update({uid: [actor["nombre"]] for uid, actor in roster.items()})
modelo = con_postura if usar_postura else simple
salida, ranking, ruta = modelo.classify(None, frase, aliases, roster, aqui, False)
print("Frase codificada e IDs:", features(frase, aliases, aqui))
print("Decisión:", salida)
print("Puntuaciones de la ruta:", ranking)
print("Contrato para Unreal:", json.dumps(wire(salida), ensure_ascii=False, indent=2))

Frase codificada e IDs: ('unidad avancen al lugar', ['Alfa-2'], ['centro'])
Decisión: {'tipo': 'accion', 'sujeto': {'id': 'Alfa-2', 'tipo': 'soldado', 'nombre': 'Juan'}, 'lugar': 'centro', 'accion': 'avanzar'}
Puntuaciones de la ruta: [('movimiento', 1.0), ('charla', 0.0)]
Contrato para Unreal: {
  "version": 1,
  "intent": "orden",
  "subject": {
    "type": "soldado",
    "name": "Alfa-2"
  },
  "action": "avanzar",
  "location": {
    "id": "centro"
  },
  "target": null,
  "model_confidence": null
}
